# 04 · Değerlendirme (Evaluation)

Kaydedilmiş en iyi model, eğitim ve model seçiminde **hiç kullanılmamış** kayıt-bazlı test bölümünde
değerlendirilir.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "passive_radar").is_dir())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import classification_report

from passive_radar.config import ProjectConfig
from passive_radar.data.dataset import RangeDopplerDataset, make_data_loader
from passive_radar.data.indexing import load_split_csv, records_in_split, scan_unlabeled_directory
from passive_radar.data.preprocessing import load_and_preprocess
from passive_radar.evaluation.metrics import collect_predictions, metrics_from_predictions
from passive_radar.evaluation.plots import plot_confusion_matrix, plot_image_grid, plot_training_history
from passive_radar.inference.predictor import RangeDopplerPredictor
from passive_radar.models.checkpoint import load_checkpoint
from passive_radar.training.runtime import resolve_device
from passive_radar.training.trainer import TrainingHistory

config = ProjectConfig.default(PROJECT_ROOT)
figures_dir = config.paths.outputs_dir / "figures"
device = resolve_device(config.training.device)
checkpoint = load_checkpoint(config.paths.checkpoint_path, device=device)
print("classes:", checkpoint.class_names, "| checkpoint metadata:",
      {k: v for k, v in checkpoint.metadata.items() if k != "training_config"})

## 1. Test bölümünde bağımsız değerlendirme

In [ ]:
records = load_split_csv(config.paths.splits_csv, relative_to=config.paths.project_root)
results = {}
for split in ("val", "test"):
    split_records = records_in_split(records, split)
    dataset = RangeDopplerDataset(split_records, checkpoint.preprocessing_config)
    loader = make_data_loader(dataset, config.training.batch_size, shuffle=False)
    predictions = collect_predictions(checkpoint.model, loader, device)
    results[split] = (split_records, predictions, metrics_from_predictions(predictions))

metrics_table = pd.DataFrame({s: {k: v for k, v in m.items() if k != "confusion_matrix"}
                              for s, (_, _, m) in results.items()}).T
metrics_table

In [ ]:
test_records, test_predictions, test_metrics = results["test"]
print(classification_report(test_predictions.y_true, test_predictions.y_pred,
                            target_names=list(checkpoint.class_names), digits=3))
config.paths.outputs_dir.mkdir(parents=True, exist_ok=True)
(config.paths.outputs_dir / "test_metrics.json").write_text(json.dumps(
    {split: m for split, (_, _, m) in results.items()}, indent=2), encoding="utf-8")
print("saved outputs/test_metrics.json")

## 2. Confusion matrix

In [ ]:
plot_confusion_matrix(test_metrics["confusion_matrix"], checkpoint.class_names,
                      save_path=figures_dir / "confusion_matrix_test.png")
plt.show()

## 3. Eğitim grafikleri

In [ ]:
history = TrainingHistory.load_json(config.paths.outputs_dir / "training_history.json")
plot_training_history(history)
plt.show()

## 4. Hatalı sınıflandırılan örnekler

In [ ]:
names = checkpoint.class_names
errors = pd.DataFrame({
    "file": [r.path.name for r in test_records],
    "recording": [r.source_id for r in test_records],
    "true": [names[i] for i in test_predictions.y_true],
    "predicted": [names[i] for i in test_predictions.y_pred],
    "p_target": test_predictions.positive_probability.round(3),
})
wrong = errors[errors["true"] != errors["predicted"]]
print(f"misclassified: {len(wrong)} / {len(errors)}")
print(errors.assign(correct=errors["true"] == errors["predicted"])
      .groupby("recording")["correct"].agg(["size", "mean"]).sort_values("mean").head(10))

In [ ]:
shown = wrong.reindex((wrong["p_target"] - 0.5).abs().sort_values(ascending=False).index).head(6)
if shown.empty:
    print("No misclassified test images.")
else:
    lookup = {r.path.name: r.path for r in test_records}
    plot_image_grid(
        [load_and_preprocess(lookup[f], checkpoint.preprocessing_config) for f in shown["file"]],
        [f"true={t} pred={p} p_target={q:.2f}\n{f[:60]}" for f, t, p, q in
         shown[["file", "true", "predicted", "p_target"]].itertuples(index=False)],
        columns=2, save_path=figures_dir / "misclassified_test.png",
    )
    plt.show()

## 5. Tek görsel tahmini ve etiketsiz `data/test` demosu

In [ ]:
predictor = RangeDopplerPredictor(config.paths.checkpoint_path, device=config.training.device)
unlabeled = scan_unlabeled_directory(config.paths.unlabeled_dir)
print(predictor.predict(unlabeled[0].path))

labeled_by_hash = {r.md5: r for r in records}
demo = pd.DataFrame([
    {"file": r.path.name[-24:], **{f"p_{k}": round(v, 3) for k, v in result.probabilities.items()},
     "prediction": result.predicted_class,
     "known label (identical labelled copy)": (names[labeled_by_hash[r.md5].label] if r.md5 in labeled_by_hash else "-")}
    for r in unlabeled for result in [predictor.predict(r.path)]
])
print(demo["prediction"].value_counts().to_string())
demo.head(10)

`data/test` etiketsizdir; bu tablo yalnızca tahmin akışını gösterir. Bazı görsellerin birebir aynısı etiketli
veride bulunduğundan o satırlar için bilinen etiket gösterilir (bunlar kayıt bazlı bölmede test bölümüne düşer,
eğitimde kullanılmamıştır).

## 6. Sonuçlar ve sınırlılıklar

* Metrikler yukarıdaki tabloda ve `outputs/test_metrics.json` dosyasındadır; %80 doğruluk hedef olabilir ama garanti
  değildir. Test bölümü ~110 görsel / ~27 kayıttan oluşur → metriklerin belirsizliği yüksektir (± birkaç puan).
* Veri simülasyondur (ses kayıtları aydınlatıcı, yapay hedef yankısı); gerçek pasif radar verisine genelleme
  doğrulanmamıştır.
* Model, Plotly figürlerinin renk haritası (jet) görüntüsünü öğrenir; renk ölçeği görsel başına otomatik ayarlıysa
  mutlak güç bilgisi kaybolur. Ham Range-Doppler matrisleri mevcutsa doğrudan onlarla çalışmak daha doğru olur.
* Model yalnızca hedef var/yok kararı verir; hedefin menzili, hızı veya konumu tahmin edilmez.
* MPS üzerinde eğitim tam deterministik olmayabilir; tekrar çalıştırmalarda küçük farklar beklenir.